# Modele Rekurencyjne

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
!unzip pakiet.zip

Archive:  pakiet.zip
   creating: pakiet/
  inflating: __MACOSX/._pakiet       
  inflating: pakiet/train.pkl        
  inflating: __MACOSX/pakiet/._train.pkl  
  inflating: pakiet/.DS_Store        
  inflating: __MACOSX/pakiet/._.DS_Store  
  inflating: pakiet/test_no_target.pkl  
  inflating: __MACOSX/pakiet/._test_no_target.pkl  
  inflating: pakiet/treść_zadania.txt  
  inflating: __MACOSX/pakiet/._treść_zadania.txt  


In [4]:
import pickle
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from utils import pad_collate, train_composer_classifier, evaluate_accuracy
from model import LSTMComposerClassifier

In [11]:
from torch.utils.data import DataLoader, random_split

FOLDER_DIR = "pakiet/"

with open(FOLDER_DIR + "train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open(FOLDER_DIR + "test_no_target.pkl", "rb") as f:
    test_data = pickle.load(f)

# Split train into train/validation
train_size = int(0.8 * len(train_data))
valid_size = len(train_data) - train_size

train_dataset, valid_dataset = random_split(
    train_data,
    [train_size, valid_size]
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_collate
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_collate
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_collate
)

In [12]:
# Scan the loader to find the exact range of your raw chord data
absolute_min = 0
absolute_max = 0

for x_batch, _, _ in train_loader:
    b_min = x_batch.min().item()
    b_max = x_batch.max().item()
    if b_min < absolute_min: absolute_min = b_min
    if b_max > absolute_max: absolute_max = b_max

print(f"Raw data minimum index: {absolute_min}") # Probably -1
print(f"Raw data maximum index: {absolute_max}") # Probably 191


Raw data minimum index: 0
Raw data maximum index: 192


In [13]:
x_padded, lengths, y = next(iter(train_loader))

print("x_padded shape:", x_padded.shape)
print("lengths shape:", lengths.shape)
print("labels shape:", y.shape)

print("x_padded:", x_padded)

vocab_size = int(x_padded.max() + 2)
print("vocab_size:", vocab_size)

x_padded shape: torch.Size([32, 1344])
lengths shape: torch.Size([32])
labels shape: torch.Size([32])
x_padded: tensor([[ 81,  81,  81,  ...,  29,  13,  13],
        [145, 145, 145,  ...,   0,   0,   0],
        [  0,   0,   0,  ...,   0,   0,   0],
        ...,
        [178, 178,  74,  ...,   0,   0,   0],
        [ 74,  48,  13,  ...,   0,   0,   0],
        [145, 147,  51,  ...,   0,   0,   0]])
vocab_size: 194


In [14]:
VOCAB_SIZE = vocab_size
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 2
OUT_SIZE = 5

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMComposerClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, OUT_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history_losses = []

In [16]:
train_composer_classifier(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    valid_loader=valid_loader,
    losses_list=history_losses,
    epochs=50,
    patience=4,
    save_path="best_composer_model.pt",
)


Saved best model -> best_composer_model.pt
[1/2] Train Loss: 1.2903 | Valid Loss: 1.2178 | Valid Acc: 58.67%
[2/2] Train Loss: 1.2631 | Valid Loss: 1.2241 | Valid Acc: 57.31%
Best validation loss: 1.2178


In [17]:
model.load_state_dict(torch.load("best_composer_model.pt"))
model.eval()

LSTMComposerClassifier(
  (embedding): Embedding(194, 64, padding_idx=2)
  (lstm): LSTM(64, 128, num_layers=2)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

In [18]:
train_accuracy = evaluate_accuracy(model, train_loader, device)
print(f"\nFinal Training Accuracy: {train_accuracy:.2f}%")


Final Training Accuracy: 55.72%
